# PS S6E6 051 shared001 updated v2 RealMLP PyTorch as-is

Experiment: `exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is`

Purpose:
- Run the further-updated shared001 RealMLP PyTorch source as-is.
- Use the 2026-06-15 experiment date.
- Keep the source model/FE/training logic aligned with the uploaded updated notebook.
- Add the standard generated-data section: OOF/pred `.npy`, OOF/pred `.csv`, submission, `cv_result`, fold scores, feature metadata, class distribution, generated-data manifest, registry, and `memo.yml`.

Reference:
- User reported updated shared001 source Public LB reference: `0.97055`.


In [1]:
import os
import gc
import math
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import yaml
except Exception:
    yaml = None

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

print("PyTorch version:", torch.__version__)

def seed_everything(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)


PyTorch version: 2.10.0+cu128
device: cuda


In [2]:
# ============================================================
# 0. Configuration
# ============================================================

EXP_ID = "exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is"
COMPETITION = "PS S6E6 Predicting Stellar Class"
CREATED_AT = "2026-06-15"

ID = "id"
TARGET = "class"
CLASS_NAMES = ["GALAXY", "QSO", "STAR"]
TARGET_MAP = {"GALAXY": 0, "QSO": 1, "STAR": 2}
INV_TARGET_MAP = {v: k for k, v in TARGET_MAP.items()}

# Resolve the correct Playground Series S6E6 competition dataset.
# Do not rely on unrelated input datasets accidentally attached to the notebook.
DATA_PATH_CANDIDATES = [
    Path("/kaggle/input/playground-series-s6e6"),
    Path("/kaggle/input/competitions/playground-series-s6e6"),
]

PATH = None
for candidate in DATA_PATH_CANDIDATES:
    if (candidate / "train.csv").exists() and (candidate / "test.csv").exists() and (candidate / "sample_submission.csv").exists():
        PATH = candidate
        break

if PATH is None:
    raise FileNotFoundError(
        "Could not find PS S6E6 dataset. Expected train.csv/test.csv/sample_submission.csv in one of: "
        + ", ".join(str(p) for p in DATA_PATH_CANDIDATES)
    )

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR
OUTDIR.mkdir(parents=True, exist_ok=True)

OOF_NPY = OUTDIR / f"oof_{EXP_ID}_proba.npy"
PRED_NPY = OUTDIR / f"pred_{EXP_ID}_proba.npy"
OOF_CSV = OUTDIR / f"oof_{EXP_ID}.csv"
PRED_CSV = OUTDIR / f"pred_{EXP_ID}.csv"
SUBMISSION_PATH = OUTDIR / f"submission_{EXP_ID}.csv"
CV_RESULT_PATH = OUTDIR / f"cv_result_{EXP_ID}.json"
FOLD_SCORES_PATH = OUTDIR / f"fold_scores_{EXP_ID}.csv"
FEATURE_INFO_PATH = OUTDIR / f"feature_info_{EXP_ID}.json"
FEATURE_COLUMNS_PATH = OUTDIR / f"feature_columns_{EXP_ID}.csv"
CLASS_DISTRIBUTION_PATH = OUTDIR / f"class_distribution_{EXP_ID}.csv"
GENERATED_DATA_MANIFEST_PATH = OUTDIR / f"generated_data_manifest_{EXP_ID}.json"

print("EXP_ID:", EXP_ID)
print("PATH:", PATH)
print("OUTDIR:", OUTDIR)


EXP_ID: exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is
PATH: /kaggle/input/competitions/playground-series-s6e6
OUTDIR: /kaggle/working


## Load Data

In [3]:
train = pd.read_csv(PATH / "train.csv")
test = pd.read_csv(PATH / "test.csv")
sample_submission = pd.read_csv(PATH / "sample_submission.csv")

required_train_cols = {ID, TARGET}
required_test_cols = {ID}
assert required_train_cols.issubset(train.columns), f"train.csv missing required columns: {required_train_cols - set(train.columns)}"
assert required_test_cols.issubset(test.columns), f"test.csv missing required columns: {required_test_cols - set(test.columns)}"
assert list(sample_submission.columns) == [ID, TARGET], f"unexpected sample_submission columns: {list(sample_submission.columns)}"
assert set(train[TARGET].unique()).issubset(set(CLASS_NAMES)), f"unexpected target labels: {train[TARGET].unique()}"

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sample submission shape:", sample_submission.shape)
print("Target distribution:")
display(train[TARGET].value_counts().reindex(CLASS_NAMES))
display(train.head())


Train shape: (577347, 12)
Test shape : (247435, 11)
Sample submission shape: (247435, 2)
Target distribution:


class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


## Preprocess Features

In [4]:
%%time
train[TARGET] = train[TARGET].map(TARGET_MAP)
X = train.drop([ID, TARGET], axis=1); train_id = train[ID].copy()
y = train[TARGET].copy()
X_test = test.drop([ID], axis=1); test_id = test[ID].copy()
del train, test
gc.collect()
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
color_pairs = [
    ('u', 'g'),
    ('g', 'r'),
    ('i', 'z'),
    ('r', 'z'),
    ('u', 'z'),
]
important_combos = [
    ('alpha_cat_', 'delta_cat_'),
    ('u_cat_', 'z_cat_'),
]
important_combos = sorted(important_combos)
def feature_engineering(df, fit=False):
    # Arithmetic interaction
    df['_g_/_redshift'] = (df['g'] / (df['redshift'] + 1e-6)).astype('float32')
    df['_i_/_redshift'] = (df['i'] / (df['redshift'] + 1e-6)).astype('float32')
    df['_Distance_Modulus'] = (6.0 * np.log10(np.abs(df['redshift']) + 1e-6)).astype('float32')
    for a, b in color_pairs:
        df[f"_{a}-{b}"] = (df[a] - df[b]).astype('float32')

    # Categorize string cats
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')

    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Discretize numericals
    bin_config = {'delta': [100, 500]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype('category')

    # Create interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, new_cat_cols, new_num_cols, combo_names = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

cat_cols = sorted(cat_cols)
X = X.reindex(sorted(X.columns), axis=1)
X_test = X_test.reindex(sorted(X_test.columns), axis=1)
print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape, "\n")

FEATURE_COLUMNS = X.columns.tolist()

feature_info = {
    "n_features": int(len(FEATURE_COLUMNS)),
    "n_cat_features": int(len(cat_cols)),
    "n_num_features_initial": int(len(num_cols)),
    "cat_cols": cat_cols,
    "combo_names": combo_names,
    "color_pairs": color_pairs,
    "important_combos": important_combos,
    "source_update_note": "Further updated shared001 RealMLP source as of 2026-06-15; includes _Distance_Modulus and u-z color pair.",
}
pd.DataFrame({"feature": FEATURE_COLUMNS}).to_csv(FEATURE_COLUMNS_PATH, index=False)
with open(FEATURE_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_info, f, indent=2, ensure_ascii=False, default=str)
print("feature_info saved:", FEATURE_INFO_PATH)


X      init shape: (577347, 10)
X_test init shape: (247435, 10) 

init len(cat_cols): 2
init len(num_cols): 8 

len(new_cat_cols): 12
len(new_num_cols): 8 

prep len(cat_cols): 14
prep len(num_cols): 16 

X      prep shape: (577347, 30)
X_test prep shape: (247435, 30) 

feature_info saved: /kaggle/working/feature_info_exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is.json
CPU times: user 1.35 s, sys: 325 ms, total: 1.67 s
Wall time: 1.7 s


## Model Components

In [5]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    """
    Applies a configurable sequence of numerical transforms from CONFIG["tfms"].
    Supported: 'median_center', 'robust_scale', 'smooth_clip', 'l2_normalize'.
    'one_hot' and 'embedding' are recognised but skipped (handled by the model).
    """
    def __init__(self, tfms):
        self._tfms = [t for t in tfms
                      if t in ("median_center", "robust_scale", "smooth_clip", "l2_normalize")]

    def fit(self, X: np.ndarray, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q_diff = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            zero_idx = q_diff == 0.0
            q_diff[zero_idx] = 0.5 * (X.max(axis=0)[zero_idx] - X.min(axis=0)[zero_idx])
            self._iqr_factors = 1.0 / (q_diff + 1e-30)
            self._iqr_factors[q_diff == 0.0] = 0.0
        return self

    def transform(self, X: np.ndarray, y=None) -> np.ndarray:
        X = X.copy().astype(np.float32)
        for tfm in self._tfms:
            if tfm == "median_center":
                X -= self._median[None, :]
            elif tfm == "robust_scale":
                X *= self._iqr_factors[None, :]
            elif tfm == "smooth_clip":
                X = X / np.sqrt(1 + (X / 3) ** 2)
            elif tfm == "l2_normalize":
                norms = np.linalg.norm(X, axis=1, keepdims=True)
                X /= np.where(norms == 0, 1.0, norms)
        return X

# ── Model components ──────────────────────────────────────────────────────────
class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens: int, cat_dims, embed_dim: int = 8,
                 onehot_thresh: int = 8, device=None):
        super().__init__()
        self.n_ens = n_ens
        self.cat_dims = cat_dims
        self.onehot_features = []
        self.embed_layers = nn.ModuleList()
        self._embed_feature_indices = []

        for i, dim in enumerate(cat_dims):
            if dim <= onehot_thresh:
                self.onehot_features.append(i)
            else:
                emb = nn.ModuleList(
                    [nn.Embedding(dim, embed_dim) for _ in range(n_ens)]
                )
                self.embed_layers.append(emb)
                self._embed_feature_indices.append(i)

    def forward(self, x):
        # x: (batch, n_ens, n_cat)
        batch_size, n_ens, _ = x.shape
        features = []

        if self.onehot_features:
            onehot_x    = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_oh    = sum(onehot_dims)
            encoded     = torch.zeros(batch_size, n_ens, total_oh, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx : idx + 1].long()
                encoded.scatter_(2, pos + start, 1.0)
                start += dim
            features.append(encoded)

        for emb_list, feat_idx in zip(self.embed_layers, self._embed_feature_indices):
            feat_embs = []
            for model_idx in range(self.n_ens):
                indices = x[:, model_idx, feat_idx : feat_idx + 1].long()
                feat_embs.append(emb_list[model_idx](indices))    # (batch, 1, embed_dim)
            feat_combined = torch.cat(feat_embs, dim=1)           # (batch, n_ens, embed_dim)
            features.append(feat_combined)

        return torch.cat(features, dim=2)


class ScalingLayer(nn.Module):
    def __init__(self, n_ens: int, n_features: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None, :, :]


class NTPLinear(nn.Module):
    def __init__(self, n_ens: int, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias   = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x):
        # x: (batch, n_ens, in_features)
        # Single einsum replaces transpose → matmul → transpose
        x = torch.einsum("bki,kio->bko", x, self.weight) / math.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x


class PBLDEmbedding(nn.Module):
    """Periodic Basis with Learned Decay embedding for numerical features."""
    def __init__(self, n_ens: int, n_features: int,
                 hidden_dim: int = 16, out_dim: int = 4, freq_scale: float = 0.1,
                 activation=nn.GELU):
        super().__init__()
        self.n_ens      = n_ens
        self.n_features = n_features
        self.out_dim    = out_dim
        self.w1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim) * freq_scale)
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) / math.sqrt(hidden_dim))
        self.b2 = nn.Parameter(torch.zeros(n_ens, n_features, out_dim - 1))        
        self.act = activation()
        nn.init.uniform_(self.b1, -math.pi, math.pi)

    def forward(self, x):
        # x: (batch, n_ens, n_features)
        # All operations are fully vectorised over n_features — no Python loop.

        # (batch, n_ens, n_features, 1) * (n_ens, n_features, hidden)
        # → periodic: (batch, n_ens, n_features, hidden)
        periodic = torch.cos(
            2 * math.pi * (
                x.unsqueeze(-1) * self.w1.unsqueeze(0)   # Broadcast over batch
                + self.b1.unsqueeze(0)
            )
        )

        # (batch, n_ens, n_features, hidden) @ (n_ens, n_features, hidden, out-1)
        # → transformed: (batch, n_ens, n_features, out-1)
        transformed = self.act(
            torch.einsum("bkfh,kfhd->bkfd", periodic, self.w2)
            + self.b2.unsqueeze(0)
        )

        # Concatenate raw feature (residual) with transformed output
        # x: (batch, n_ens, n_features) → unsqueeze → (batch, n_ens, n_features, 1)
        feat = torch.cat([x.unsqueeze(-1), transformed], dim=-1)
        # (batch, n_ens, n_features, out_dim) → flatten last two dims
        # → (batch, n_ens, n_features * out_dim)
        return feat.flatten(start_dim=2)

# ── Model ─────────────────────────────────────────────────────────────
class RealMLP(nn.Module):
    def __init__(self, output_dim: int, cat_dims, n_numerical: int, cfg: dict):
        super().__init__()
        n_ens      = cfg["n_ens"]
        embed_dim  = cfg["embed_dim"]
        self.n_ens = n_ens

        self.cate = CategoricalFeatureLayer(
            n_ens=n_ens, cat_dims=cat_dims, embed_dim=embed_dim,
            onehot_thresh=cfg["onehot_thresh"],
        )
        self.num_embed = PBLDEmbedding(
            n_ens=n_ens,
            n_features=n_numerical,
            hidden_dim=cfg["pbld_hidden_dim"],
            out_dim=cfg["pbld_out_dim"],
            freq_scale=cfg["pbld_freq_scale"],
            activation=cfg["pbld_activation"],
        )

        num_emb_dim = n_numerical * cfg["pbld_out_dim"]
        cat_emb_dim = sum(
            c if c <= cfg["onehot_thresh"] else embed_dim for c in cat_dims
        )
        total_dim = num_emb_dim + cat_emb_dim
        hidden_dims = cfg["hidden_dims"]

        act = cfg["activation"]

        # Build layers, tracking which NTPLinear is the "first layer"
        # so we can give it a separate lr group.
        # Each hidden position gets its own Dropout instance (shared instance
        # would only register once in nn.Sequential and break the scheduler).
        layers = []
        if cfg["add_front_scale"]:
            layers.append(ScalingLayer(n_ens=n_ens, n_features=total_dim))

        self._dropout_modules = []    # Kept for live p_drop_sched updates
        in_dim = total_dim
        for i, out_dim_h in enumerate(hidden_dims):
            linear = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=out_dim_h)
            if i == 0:
                self.first_linear = linear    # Reference for separate lr group
            drop = nn.Dropout(cfg["dropout"])
            self._dropout_modules.append(drop)
            layers += [linear, act(), drop]
            in_dim = out_dim_h

        self.hidden = nn.Sequential(*layers)
        self.output_layer = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=output_dim)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)
        combined = torch.cat([x_num, x_cat], dim=2)
        x = self.hidden(combined)
        x = self.output_layer(x)
        return F.softmax(x, dim=2)    # (batch, n_ens, output_dim)

# ── Schedule helpers ─────────────────────────────────────────────────────────
def apply_schedule(init_value: float, progress: float, sched: str,
                   flat_ratio: float = 0.3) -> float:
    """
    Supported schedules:
      'constant'    – no decay
      'cos'         – cosine from init to 0
      'flat_cos'    – flat for flat_ratio, then cosine to 0
      'flat_anneal' – flat for flat_ratio, then linear to 0
      'sqrt_cos'    – sqrt of cosine annealing (slower decay)
      'expm4t'      – exponential decay: init * exp(-4 * progress)
    """
    if sched == "constant":
        return init_value
    elif sched == "cos":
        return init_value * (math.cos(math.pi * progress) + 1) / 2
    elif sched == "flat_cos":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (math.cos(math.pi * t) + 1) / 2
    elif sched == "flat_anneal":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (1 - t)
    elif sched == "sqrt_cos":
        return init_value * math.sqrt((math.cos(math.pi * progress) + 1) / 2)
    elif sched == "expm4t":
        return init_value * math.exp(-4 * progress)
    else:
        raise ValueError(f"Unknown schedule: '{sched}'")

# ── Per-parameter-group builder ───────────────────────────────────────────────
def get_parameter_groups(model: RealMLP, p: dict):
    """
    Five groups with independent lr / wd:
      0 – ScalingLayer params  (scale.*)
      1 – PBLD / num_embed params
      2 – first hidden linear weight
      3 – all other weights
      4 – all biases (excluding those already in groups 0-2)
    Note: PBLD has its own bias params (b1, b2) which belong to group 1, not 4.
    The ordering of checks therefore is: scale → num_embed → first_w → bias → other_w.
    """
    first_linear_weight_id = id(model.first_linear.weight)

    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if "num_embed" in name:
            # All PBLD params (weights and biases) get their own lr group
            pbld_p.append(param)
        elif "scale" in name:
            scale_p.append(param)
        elif id(param) == first_linear_weight_id:
            first_w_p.append(param)
        elif "bias" in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)

    LR = p["lr"]
    WD = p["weight_decay"]
    return [
        {"params": scale_p,   "lr": LR * p["lr_scale_mult"],         "weight_decay":  WD * p["wd_scale_mult"],         "group": "scale"},
        {"params": pbld_p,    "lr": LR * p["pbld_lr_factor"],        "weight_decay":  WD,                              "group": "pbld"},
        {"params": first_w_p, "lr": LR * p["first_layer_lr_factor"], "weight_decay":  WD * p["first_layer_wd_factor"], "group": "first_w"},
        {"params": other_w_p, "lr": LR,                              "weight_decay":  WD,                              "group": "other_w"},
        {"params": bias_p,    "lr": LR * p["lr_bias_mult"],          "weight_decay":  WD * p["wd_bias_mult"],          "group": "bias"},
    ]

# ── Multiclass label-smoothed cross-entropy with optional class weights ───────
def smooth_ce_loss(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    ls: float = 0.0,
    class_weights: torch.Tensor = None,
) -> torch.Tensor:
    """
    y_true        : (N,)    long
    y_pred        : (N, C)  probabilities
    class_weights : (C,)    per-class weight tensor (optional)
    """
    n_classes = y_pred.size(1)
    y_smooth  = torch.full_like(y_pred, ls / n_classes)
    y_smooth.scatter_(1, y_true.unsqueeze(1), 1.0 - ls + ls / n_classes)
    per_sample_loss = -(y_smooth * torch.log(y_pred.clamp(1e-15, 1))).sum(dim=1)
    if class_weights is not None:
        sample_weights = class_weights[y_true]
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum()
    return per_sample_loss.mean()

# ── Sklearn-compatible wrapper ────────────────────────────────────────────────
class RealMLP_TD_Classifier(BaseEstimator):
    """
    Sklearn-compatible wrapper around RealMLP, matching the interface of
    pytabkit's RealMLP_TD_Classifier:
    model = RealMLP_TD_Classifier(**CONFIG)
    model.fit(X_train, y_train, X_val, y_val, cat_col_names=CATS)
    proba = model.predict_proba(X_test)
    """
    def __init__(self, **kwargs):
        # Accept any subset of CONFIG keys; fall back to CONFIG defaults
        self.params = {**CONFIG, **kwargs}

    def fit(self, X_train: pd.DataFrame, y_train, X_val: pd.DataFrame, y_val,
            cat_col_names=None, X_test: pd.DataFrame = None):
        p   = self.params
        dev = torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        verbose = p["verbosity"]
        cat_col_names = cat_col_names or []
        num_col_names = [c for c in X_train.columns if c not in cat_col_names]

        # ── Split num / cat ──────────────────────────────────────────────────
        X_tr_num  = X_train[num_col_names].values.astype(np.float32)
        X_val_num = X_val[num_col_names].values.astype(np.float32)
        X_tr_cat  = X_train[cat_col_names].values.astype(np.int64)
        X_val_cat = X_val[cat_col_names].values.astype(np.int64)
        y_tr      = np.asarray(y_train)
        y_v       = np.asarray(y_val)

        # ── Numerical preprocessing ───────────────────────────────────────────
        self.preprocessor_ = NumericalPreprocessor(p["tfms"])
        self.preprocessor_.fit(X_tr_num)
        X_tr_num  = self.preprocessor_.transform(X_tr_num)
        X_val_num = self.preprocessor_.transform(X_val_num)

        # ── Categorical dims ─────────────────────────────────────────────────────────
        self.cat_col_names_ = cat_col_names
        self.num_col_names_ = num_col_names
        if cat_col_names:
            all_cat = [X_tr_cat, X_val_cat]
            if X_test is not None:
                all_cat.append(X_test[cat_col_names].values.astype(np.int64))
            cat_dims = (np.concatenate(all_cat, axis=0).max(axis=0) + 1).tolist()
        else:
            cat_dims = []
        self.cat_dims_ = cat_dims

        # Clamp indices to [0, dim-1] — -1 codes (unseen pandas categories)
        # wrap to huge ints on GPU and cause device-side assert
        if cat_dims:
            cat_max = np.array(cat_dims) - 1
            X_tr_cat  = np.clip(X_tr_cat,  0, cat_max)
            X_val_cat = np.clip(X_val_cat, 0, cat_max)

        # ── Class weights ────────────────────────────────────────────────────
        classes       = np.unique(y_tr)
        self.classes_ = classes
        weights_np    = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weights = torch.as_tensor(weights_np, dtype=torch.float32, device=dev)

        # ── Build model ──────────────────────────────────────────────────────
        n_classes    = len(classes)
        self.model_  = RealMLP(
            output_dim=n_classes,
            cat_dims=cat_dims,
            n_numerical=X_tr_num.shape[1],
            cfg=p,
        ).to(dev)

        param_groups = get_parameter_groups(self.model_, p)
        # Store the base lr on each group so the scheduler can scale from it
        for g in param_groups:
            g["lr_base"] = g["lr"]
        optimizer = torch.optim.AdamW(
            param_groups,
            betas=(p["mom"], p["sq_mom"]),
        )

        # ── To tensors ───────────────────────────────────────────────────────
        Xtn = torch.as_tensor(X_tr_num,  dtype=torch.float32, device=dev)
        Xtc = torch.as_tensor(X_tr_cat,  dtype=torch.long,    device=dev)
        ytt = torch.as_tensor(y_tr,      dtype=torch.long,    device=dev)
        Xvn = torch.as_tensor(X_val_num, dtype=torch.float32, device=dev)
        Xvc = torch.as_tensor(X_val_cat, dtype=torch.long,    device=dev)

        n_ens       = p["n_ens"]
        train_bs    = p["train_bs"]
        eval_bs     = p["eval_bs"]
        epochs      = p["epochs"]
        lr_sched    = p["lr_sched"]
        flat_ratio  = p["flat_ratio"]
        ema_decay   = p["ema_decay"]
        total_steps = epochs * len(y_tr)
        train_order = np.arange(len(y_tr))

        best_score      = -np.inf
        best_epoch      = 0
        best_val_probs  = None
        best_state      = None
        ema_state       = None
        if ema_decay > 0:
            ema_state = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}

        # ── Epoch loop ───────────────────────────────────────────────────────
        for epoch in range(epochs):
            self.model_.train()
            for start in range(0, len(y_tr), train_bs):
                progress  = (epoch * len(y_tr) + start) / total_steps
                idx_batch = train_order[start : start + train_bs]

                # Update lr for each param group using its base lr
                for g in optimizer.param_groups:
                    g["lr"] = apply_schedule(g["lr_base"], progress, lr_sched, flat_ratio)

                optimizer.zero_grad()
                y_pred = self.model_(Xtn[idx_batch], Xtc[idx_batch])    # (bs, n_ens, C)

                ls_val   = apply_schedule(p["ls_eps"],  progress, p["ls_eps_sched"],  flat_ratio)
                drop_val = apply_schedule(p["dropout"], progress, p["p_drop_sched"],  flat_ratio)
                for dm in self.model_._dropout_modules:
                    dm.p = drop_val

                loss = smooth_ce_loss(
                    ytt[idx_batch].repeat_interleave(n_ens),
                    y_pred.reshape(-1, n_classes),
                    ls=ls_val,
                    class_weights=class_weights,
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), p["grad_clip"])
                optimizer.step()

                if ema_state is not None:
                    with torch.no_grad():
                        model_state = self.model_.state_dict()
                        for key, value in model_state.items():
                            if torch.is_floating_point(value):
                                ema_state[key].mul_(ema_decay).add_(value.detach(), alpha=1.0 - ema_decay)
                            else:
                                ema_state[key].copy_(value)

            np.random.shuffle(train_order)

            # ── Validation ───────────────────────────────────────────────────
            self.model_.eval()

            live_state = None
            if ema_state is not None:
                live_state = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}
                self.model_.load_state_dict(ema_state, strict=True)
            
            with torch.no_grad():
                val_probs = np.concatenate([
                    self.model_(Xvn[s : s + eval_bs], Xvc[s : s + eval_bs])
                        .mean(dim=1).cpu().numpy()
                    for s in range(0, len(y_v), eval_bs)
                ], axis=0)

            epoch_score = balanced_accuracy_score(y_v, np.argmax(val_probs, axis=1))
            improved    = epoch_score > best_score
            if improved:
                best_score     = epoch_score
                best_epoch     = epoch + 1
                best_val_probs = val_probs.copy()
                state_src      = ema_state if ema_state is not None else self.model_.state_dict()
                best_state     = {k: v.detach().clone() for k, v in state_src.items()}

            if verbose >= 2:
                print(
                    f"  epoch {epoch + 1}/{epochs}  "
                    f"score = {epoch_score:.5f}  "
                    f"best = {best_score:.5f}  "
                    f"ls = {ls_val:.4f}  drop = {drop_val:.4f}"
                    + (" ✓" if improved else "")
                )

            # ── Early stopping ────────────────────────────────────────────────
            if p["use_early_stopping"]:
                patience = (best_epoch * p["early_stopping_multiplicative_patience"]
                            + p["early_stopping_additive_patience"])
                if (epoch + 1) > patience:
                    if verbose >= 1:
                        print(f"  Early stopping at epoch {epoch + 1} "
                              f"(best epoch {best_epoch})")
                    break

        # ── Restore best weights ──────────────────────────────────────────────
        if best_state is not None:
            self.model_.load_state_dict(best_state, strict=True)
        
        self.best_score_     = best_score
        self.best_val_probs_ = best_val_probs
        self._dev            = dev
        if verbose >= 1:
            print(f"  → best score: {best_score:.5f}  (epoch {best_epoch})")
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        eval_bs = self.params["eval_bs"]
        X_num = self.preprocessor_.transform(
            X[self.num_col_names_].values.astype(np.float32)
        )
        X_cat = X[self.cat_col_names_].values.astype(np.int64)
        # Clamp to valid embedding range — guards against -1 codes (unseen
        # categories in pandas category dtype) which wrap to large ints on GPU
        X_cat = np.clip(X_cat, 0, np.array(self.cat_dims_) - 1)
        Xn = torch.as_tensor(X_num, dtype=torch.float32, device=self._dev)
        Xc = torch.as_tensor(X_cat, dtype=torch.long,    device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            return np.concatenate([
                self.model_(Xn[s : s + eval_bs], Xc[s : s + eval_bs])
                    .mean(dim=1).cpu().numpy()
                for s in range(0, len(X_num), eval_bs)
            ], axis=0)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

## Parameter Configuration

In [6]:
CONFIG = {
    # --- Model architecture ---
    "n_ens":         8,                # Number of ensemble members inside one RealMLP
    "embed_dim":     8,                # Embedding dim for high-cardinality categoricals  7
    "onehot_thresh": 8,                # Categoricals with nunique <= this get one-hot encoded
    "hidden_dims":   [512, 512, 512],  # MLP hidden layer sizes
    "dropout":       0.06,             # Base dropout probability value
    "p_drop_sched":  "expm4t",         # 'expm4t' | 'flat_cos' | 'constant'
    "activation":    nn.SiLU,          # Activation class for MLP layers
    "add_front_scale": True,           # Whether to prepend a ScalingLayer before MLP layers

    # --- PBLD (periodic) embedding for numericals ---
    "pbld_hidden_dim": 20,        # plr_hidden_1 in official pytabkit's API
    "pbld_out_dim":    5,         # plr_hidden_2 + 1 (includes residual raw feature)
    "pbld_freq_scale": 5.0,       # plr_sigma: init std of frequency weights
    "pbld_activation": nn.PReLU,  # plr_act_name: activation inside PBLD
    "pbld_lr_factor":  0.093,     # plr_lr_factor: PBLD param lr = lr * pbld_lr_factor

    # --- Optimizer ---
    "lr":               0.01,       # Base learning rate (weights)
    "mom":              0.9,        # Adam beta1
    "sq_mom":           0.98,       # Adam beta2
    "lr_sched":        "flat_cos",  # 'flat_cos' | 'flat_anneal' | 'cos' | 'constant'
    "flat_ratio":       0.3,        # Flat fraction for flat_cos / flat_anneal schedules
    "first_layer_lr_factor": 1.0,   # lr multiplier for first MLP layer weights
    "first_layer_wd_factor": 0.1,   # wd multiplier for first MLP layer weights
    "lr_scale_mult":    10.0,       # Scale-layer lr = lr * lr_scale_mult
    "lr_bias_mult":     0.1,        # Bias        lr = lr * lr_bias_mult
    "weight_decay":     0.013,      # Base weight decay (weights)
    "wd_scale_mult":    0.1,        # Scale-layer wd = weight_decay * wd_scale_mult
    "wd_bias_mult":     0.5,        # Bias        wd = weight_decay * wd_bias_mult
    "ema_decay":        0.997875,   # Exponential Moving Average decay of model weights
    "grad_clip":        1.2,        # Graient clipping value

    # --- Label smoothing ---
    "ls_eps":       0.04,   # Base label smoothing epsilon
    "ls_eps_sched": "cos",  # 'cos' | 'sqrt_cos' | 'constant'

    # --- Preprocessing ---
    # Supported tfms (applied in order to numerical features):
    # 'median_center', 'robust_scale', 'smooth_clip', 'l2_normalize'
    # categorical tfms ('one_hot', 'embedding') are handled separately by the model
    "tfms": ["median_center", "robust_scale"],

    # --- Training loop ---
    "epochs":    6,
    "train_bs":  256,
    "eval_bs":   10240,
    "verbosity": 2,      # 0 = silent, 1 = fold summary only, 2 = per-epoch

    # --- Early stopping ---
    "use_early_stopping":                     False,
    "early_stopping_additive_patience":       10,     # stop if epoch > best * mult + add
    "early_stopping_multiplicative_patience": 1,

    # --- Device ---
    "device":       "cuda",
    "random_state": 42,
}

# --- Fold split ---
FOLDS = 5
SEED = 42
TE = True
assert CONFIG["random_state"] == SEED
print("CONFIG random_state:", CONFIG["random_state"])


CONFIG random_state: 42


## Train K-Fold

In [7]:
%%time
def metric(y_true, y_pred):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred)

def recalls_by_class(y_true, proba):
    pred = np.argmax(proba, axis=1)
    recalls = recall_score(y_true, pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {f"recall_{CLASS_NAMES[i]}": float(recalls[i]) for i in range(len(CLASS_NAMES))}

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
oof_preds = np.zeros((len(X), y.nunique()), dtype=np.float32)
test_preds = np.zeros((len(X_test), y.nunique()), dtype=np.float32)
fold_records = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_tst = X_test.copy()

    # Target encoding
    if TE:
        te_cols = combo_names
        encoder = TargetEncoder(cv=FOLDS, smooth='auto', shuffle=True, random_state=SEED)
        tr_enc = encoder.fit_transform(X_tr[te_cols], y.iloc[tr_idx])
        val_enc = encoder.transform(X_val[te_cols])
        tst_enc = encoder.transform(X_tst[te_cols])

        te_names = [f"_{col}TE_class{cls}" for col in te_cols for cls in range(y.nunique())]
        X_tr[te_names] = tr_enc
        X_val[te_names] = val_enc
        X_tst[te_names] = tst_enc

    if fold == 1:
        print("len(FEATURES):", len(X_tr.columns.tolist()), "\n")
    print("#"*16)
    print(f"### Fold {fold}/{FOLDS} ...")
    print("#"*16)

    model = RealMLP_TD_Classifier(**CONFIG)
    model.fit(
        X_tr, y.iloc[tr_idx],
        X_val, y.iloc[val_idx],
        cat_col_names=cat_cols,
    )
    oof_preds[val_idx] = model.best_val_probs_.astype(np.float32)
    test_preds += model.predict_proba(X_tst).astype(np.float32) / FOLDS

    fold_score = metric(y.iloc[val_idx], oof_preds[val_idx])
    rec = recalls_by_class(y.iloc[val_idx], oof_preds[val_idx])
    row = {
        "fold": fold,
        "balanced_accuracy": float(fold_score),
        **rec,
        "n_train": int(len(tr_idx)),
        "n_valid": int(len(val_idx)),
    }
    fold_records.append(row)
    print(f"\nFold {fold} | Score: {fold_score:.6f} | recalls: {rec}\n")
    torch.cuda.empty_cache()
    gc.collect()

fold_scores = pd.DataFrame(fold_records)
overall_cv = metric(y, oof_preds)
overall_recalls = recalls_by_class(y, oof_preds)
cm = confusion_matrix(y, np.argmax(oof_preds, axis=1), labels=[0, 1, 2])

print("="*26)
print(f"Overall OOF Score: {overall_cv:.8f}")
print("Overall recalls:", overall_recalls)
print("Confusion matrix:")
print(cm)
print("="*26, "\n")
display(fold_scores)
fold_scores.to_csv(FOLD_SCORES_PATH, index=False)


len(FEATURES): 36 

################
### Fold 1/5 ...
################
  epoch 1/6  score = 0.96688  best = 0.96688  ls = 0.0373  drop = 0.0308 ✓
  epoch 2/6  score = 0.96927  best = 0.96927  ls = 0.0300  drop = 0.0158 ✓
  epoch 3/6  score = 0.96961  best = 0.96961  ls = 0.0200  drop = 0.0081 ✓
  epoch 4/6  score = 0.97006  best = 0.97006  ls = 0.0100  drop = 0.0042 ✓
  epoch 5/6  score = 0.96997  best = 0.97006  ls = 0.0027  drop = 0.0021
  epoch 6/6  score = 0.97008  best = 0.97008  ls = 0.0000  drop = 0.0011 ✓
  → best score: 0.97008  (epoch 6)

Fold 1 | Score: 0.970077 | recalls: {'recall_GALAXY': 0.9647266080322137, 'recall_QSO': 0.976268726791583, 'recall_STAR': 0.9692354185554548}

################
### Fold 2/5 ...
################
  epoch 1/6  score = 0.96705  best = 0.96705  ls = 0.0373  drop = 0.0308 ✓
  epoch 2/6  score = 0.96832  best = 0.96832  ls = 0.0300  drop = 0.0158 ✓
  epoch 3/6  score = 0.96866  best = 0.96866  ls = 0.0200  drop = 0.0081 ✓
  epoch 4/6  score = 0.968

,fold,balanced_accuracy,recall_GALAXY,recall_QSO,recall_STAR,n_train,n_valid
0,1,0.970077,0.964727,0.976269,0.969235,461877,115470
1,2,0.968658,0.956938,0.976354,0.972681,461877,115470
2,3,0.968388,0.956382,0.974775,0.974009,461878,115469
3,4,0.968836,0.958938,0.976097,0.971472,461878,115469
4,5,0.968579,0.959336,0.976140,0.970263,461878,115469


CPU times: user 17min 55s, sys: 4.34 s, total: 18min
Wall time: 18min 7s


## Save Outputs / Generated Data / Registry / Memo

In [8]:
# ============================================================
# 7. Save outputs / generated data / registry / memo
# ============================================================

# Save probability assets for later blend/correlation/stacker checks.
# This is the generated-data section that creates all reusable artifacts.
np.save(OOF_NPY, oof_preds.astype(np.float32))
np.save(PRED_NPY, test_preds.astype(np.float32))

# OOF / pred CSVs
columns = [f"proba_{c}" for c in CLASS_NAMES]
oof_df = pd.DataFrame(oof_preds, columns=columns)
oof_df.insert(0, ID, train_id.values)
oof_df[TARGET] = y.map(INV_TARGET_MAP).values
oof_df["pred"] = [CLASS_NAMES[i] for i in np.argmax(oof_preds, axis=1)]
oof_df.to_csv(OOF_CSV, index=False)

pred_df = pd.DataFrame(test_preds, columns=columns)
pred_df.insert(0, ID, test_id.values)
pred_df.to_csv(PRED_CSV, index=False)

# Submission
sub = pd.DataFrame({ID: test_id.values, TARGET: np.argmax(test_preds, axis=1)})
sub[TARGET] = sub[TARGET].map(INV_TARGET_MAP)
sub.to_csv(SUBMISSION_PATH, index=False)

# Class distribution
class_distribution = sub[TARGET].value_counts().reindex(CLASS_NAMES, fill_value=0).rename_axis(TARGET).reset_index(name="count")
class_distribution["ratio"] = class_distribution["count"] / len(sub)
class_distribution.to_csv(CLASS_DISTRIBUTION_PATH, index=False)

fold_std = float(fold_scores["balanced_accuracy"].std(ddof=0))

cv_result = {
    "exp_id": EXP_ID,
    "competition": COMPETITION,
    "created_at": CREATED_AT,
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "model_family": "RealMLP",
    "model_type": "shared001_updated_v2_realmlp_pytorch_as_is",
    "source": "ps-s6-e6-realmlp-pytorch updated shared001 source attached by user on 2026-06-15",
    "cv_score": float(overall_cv),
    "fold_std": fold_std,
    "recalls": {k.replace("recall_", ""): v for k, v in overall_recalls.items()},
    "confusion_matrix": cm.astype(int).tolist(),
    "n_splits": FOLDS,
    "seed": SEED,
    "use_original": False,
    "use_fe": True,
    "use_bias_search": False,
    "use_external_stacking_files": False,
    "public_lb_reference_from_source": 0.97055,
    "variant_design": {
        "base_lineage": [
            "exp_20260605_015_shared001_realmlp_as_is",
            "exp_20260610_035_shared001_updated_realmlp_pytorch_as_is",
        ],
        "run_type": "as_is_updated_source",
        "update_note": "Further updated shared001 source; run as-is while adding standard generated artifacts.",
    },
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_csv": OOF_CSV.name,
        "pred_csv": PRED_CSV.name,
        "submission": SUBMISSION_PATH.name,
        "cv_result": CV_RESULT_PATH.name,
        "fold_scores": FOLD_SCORES_PATH.name,
        "feature_info": FEATURE_INFO_PATH.name,
        "feature_columns": FEATURE_COLUMNS_PATH.name,
        "class_distribution": CLASS_DISTRIBUTION_PATH.name,
        "generated_data_manifest": GENERATED_DATA_MANIFEST_PATH.name,
    },
    "config": {
        k: (str(v) if callable(v) or isinstance(v, type) else v)
        for k, v in CONFIG.items()
    },
}
with open(CV_RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(cv_result, f, indent=2, ensure_ascii=False, default=str)

# Generated data manifest for quick dataset update / upload checks
generated_data_manifest = {
    "exp_id": EXP_ID,
    "created_at": CREATED_AT,
    "purpose": "Reusable generated artifacts for blend/correlation/stacker experiments.",
    "files": cv_result["outputs"],
    "absolute_paths": {
        "oof_proba": str(OOF_NPY),
        "pred_proba": str(PRED_NPY),
        "oof_csv": str(OOF_CSV),
        "pred_csv": str(PRED_CSV),
        "submission": str(SUBMISSION_PATH),
        "cv_result": str(CV_RESULT_PATH),
        "fold_scores": str(FOLD_SCORES_PATH),
        "feature_info": str(FEATURE_INFO_PATH),
        "feature_columns": str(FEATURE_COLUMNS_PATH),
        "class_distribution": str(CLASS_DISTRIBUTION_PATH),
    },
}
with open(GENERATED_DATA_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(generated_data_manifest, f, indent=2, ensure_ascii=False, default=str)

registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "RealMLP",
    "feature_family": "shared001_updated_v2_realmlp_pytorch_fe",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(overall_cv),
    "fold_std": fold_std,
    "public_lb": np.nan,
    "public_lb_reference_from_source": 0.97055,
    "use_original": False,
    "use_fe": True,
    "use_bias_search": False,
    "oof_path": str(OOF_NPY),
    "pred_path": str(PRED_NPY),
    "submission_path": str(SUBMISSION_PATH),
    "role": "shared001_updated_v2_realmlp_material",
    "status": "completed",
    "keep_hold_drop": "pending_public_lb_and_blend_check",
    "notes": "Further updated shared001 RealMLP PyTorch as-is. Source/user-reported updated LB was 0.97055. Saves exp_id-named generated artifacts for add051 blend/stacker checks.",
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "shared001 updated v2 RealMLP PyTorch as-is",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "RealMLP_TD_Classifier PyTorch implementation",
        "created_at": CREATED_AT,
    },
    "objective": {
        "summary": (
            "Run the further-updated shared001 RealMLP PyTorch notebook as-is and save standard generated artifacts "
            "for later blend/correlation/stacker checks."
        ),
        "success_criteria": [
            "run updated shared001 source as-is",
            "save OOF proba npy with exp_id in filename",
            "save test pred proba npy with exp_id in filename",
            "save OOF/test prediction CSVs",
            "save submission",
            "save cv_result",
            "save fold_scores",
            "save feature_info and feature_columns",
            "save class_distribution",
            "save generated_data_manifest",
            "update oof_registry",
            "write memo.yml",
        ],
    },
    "data": {
        "train_path": str(PATH / "train.csv"),
        "test_path": str(PATH / "test.csv"),
        "sample_submission_path": str(PATH / "sample_submission.csv"),
        "target_col": TARGET,
        "id_col": ID,
        "use_original": False,
        "use_external_stacking_files": False,
    },
    "seed": {
        "seed": SEED,
        "random_seed": SEED,
        "numpy_seed": SEED,
        "torch_seed": SEED,
        "stratified_kfold_random_state": SEED,
        "target_encoder_random_state": SEED,
        "config_random_state": CONFIG.get("random_state"),
        "audit": "CONFIG random_state is asserted to equal SEED before training.",
    },
    "features": {
        "feature_family": "shared001_updated_v2_realmlp_pytorch_fe",
        "feature_info": feature_info,
        "target_encoder_safety": (
            "TargetEncoder is fit inside each fold on train fold combo features and labels, "
            "then transforms validation and test."
        ),
    },
    "model": {
        "family": "RealMLP",
        "type": "shared001_updated_v2_realmlp_pytorch_as_is",
        "config": cv_result["config"],
    },
    "cv": {
        "scheme": "StratifiedKFold",
        "n_splits": FOLDS,
        "shuffle": True,
        "random_state": SEED,
        "score": float(overall_cv),
        "recalls": {k.replace("recall_", ""): v for k, v in overall_recalls.items()},
        "confusion_matrix": cm.astype(int).tolist(),
        "fold_scores": fold_scores.to_dict(orient="records"),
        "fold_std": fold_std,
    },
    "outputs": {
        "oof_proba": OOF_NPY.name,
        "pred_proba": PRED_NPY.name,
        "oof_csv": OOF_CSV.name,
        "pred_csv": PRED_CSV.name,
        "submission": SUBMISSION_PATH.name,
        "cv_result": CV_RESULT_PATH.name,
        "fold_scores": FOLD_SCORES_PATH.name,
        "feature_info": FEATURE_INFO_PATH.name,
        "feature_columns": FEATURE_COLUMNS_PATH.name,
        "class_distribution": CLASS_DISTRIBUTION_PATH.name,
        "generated_data_manifest": GENERATED_DATA_MANIFEST_PATH.name,
        "registry": "oof_registry.csv",
    },
    "reference": {
        "source_notebook_public_lb_reported_by_user": 0.97055,
        "note": "Reference value from the updated shared001 source/user report; this experiment's own Public LB should be recorded after submission.",
    },
    "judgement": {
        "status": "pending_public_lb_and_blend_check",
        "role": "shared001_updated_v2_realmlp_material",
        "reason": (
            "The updated shared001 source reportedly has strong Public LB despite limited CV gain. "
            "Adoption depends on this run's CV/Public LB and blend/stacker contribution against 040/038/050/049/045/033."
        ),
        "next_action": [
            f"Submit {SUBMISSION_PATH.name}",
            "Record Public LB",
            "Add generated OOF/pred npy to npy-files dataset",
            "Run add051 blend check if LB/CV are acceptable",
            "Only run GPU LogReg stacker add051 if blend check shows meaningful contribution",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("saved:", OOF_NPY)
print("saved:", PRED_NPY)
print("saved:", SUBMISSION_PATH)
print("cv_result saved:", CV_RESULT_PATH)
print("generated manifest saved:", GENERATED_DATA_MANIFEST_PATH)
print("registry saved:", registry_path)
print("memo saved:", memo_path)
display(registry.tail())
display(class_distribution)
sub.head()


saved: /kaggle/working/oof_exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is_proba.npy
saved: /kaggle/working/pred_exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is_proba.npy
saved: /kaggle/working/submission_exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is.csv
cv_result saved: /kaggle/working/cv_result_exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is.json
generated manifest saved: /kaggle/working/generated_data_manifest_exp_20260615_051_shared001_updated_v2_realmlp_pytorch_as_is.json
registry saved: /kaggle/working/oof_registry.csv
memo saved: /kaggle/working/memo.yml


,exp_id,model_family,feature_family,cv_metric,cv_score,fold_std,public_lb,public_lb_reference_from_source,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260615_051_shared001_updated_v2_realmlp_...,RealMLP,shared001_updated_v2_realmlp_pytorch_fe,balanced_accuracy_score,0.968908,0.000602,NaN,0.97055,False,True,False,/kaggle/working/oof_exp_20260615_051_shared001...,/kaggle/working/pred_exp_20260615_051_shared00...,/kaggle/working/submission_exp_20260615_051_sh...,shared001_updated_v2_realmlp_material,completed,pending_public_lb_and_blend_check,Further updated shared001 RealMLP PyTorch as-i...


,class,count,ratio
0,GALAXY,156863,0.633956
1,QSO,51502,0.208144
2,STAR,39070,0.157900


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
